# Day 4 · Representation learning — embeddings, and their honest limits

**Today's goal:** build code and patient embeddings from the OMOP stream, see what they buy
over a bag of codes, and see concretely why they cannot be handed to an underwriter.

### Summary

An embedding turns a discrete concept into a dense vector positioned so concepts used in
similar contexts land near each other. Pool a patient's concept vectors and you get a
patient vector. That is the same object CLMBR produces — CLMBR just learns it with a
transformer over ordered timelines instead of a factorisation over co-occurrence counts.
Building the cheap version tells you exactly what the expensive version is supposed to add.

### Technique chosen per subtopic

| Subtopic | We use | Mentioned only |
|---|---|---|
| Concept embedding | **PPMI + truncated SVD** — deterministic, no GPU | word2vec/skip-gram, GloVe, med2vec |
| Patient pooling | **mean** and **recency-weighted mean** | attention pooling, RNN final state |
| Ontology structure | *mentioned* | node2vec on `concept_ancestor`, GRAM |
| Sequence model | *mentioned* | CLMBR, BEHRT, Med-BERT, CEHR-BERT, MOTOR |
| Time encoding | **exponential recency weights** | time2vec, artificial time tokens |

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 40)
import ethos_omop as eo

OUT_DIR = "../data_input"   # extract_cohort() output; use "omop_extract" for extract_omop()
DOMAINS = ['Visit', 'Condition', 'Drug', 'Procedure', 'Measurement']

if not os.path.isdir(OUT_DIR):
    raise FileNotFoundError(
        f"{OUT_DIR}/ not found. Run Day 1's eo.extract_cohort(...) (or "
        f"eo.extract_omop(...) with OUT_DIR='omop_extract') first, or "
        f"python make_demo_extract.py for a synthetic stand-in.")

concept  = eo.load_concept(OUT_DIR)
codesets = eo.build_codesets(concept, verbose=False)
events   = eo.tag_events(eo.load_events(OUT_DIR, domains=DOMAINS), codesets)
person   = eo.load_person(OUT_DIR)
truth    = eo.load_truth(OUT_DIR)                   # None against real data

print(f"events {events.shape}   patients {events.person_id.nunique()}")
print(f"codeset coverage: {events.in_codeset.mean():.1%} of events carry a clinical tag")
print(f"truth labels: {'present (DEMO extract)' if truth is not None else 'ABSENT (real data)'}")
events.head(6)

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

seqs = eo.sequences(events, col='tag')
pids = sorted(seqs)
y    = (truth.set_index('person_id').archetype.reindex(pids).values
        if truth is not None else None)

## 1 · Co-occurrence over a sliding window

For each tag, count which other tags appear within `W` positions in a patient's timeline.
This is the skip-gram objective, counted rather than learned by gradient descent.

In [ ]:
W     = 3
vocab = sorted({c for s in seqs.values() for c in s})
idx   = {c: i for i, c in enumerate(vocab)}
C     = np.zeros((len(vocab), len(vocab)))

for s in seqs.values():
    for i, c in enumerate(s):
        for j in range(max(0, i - W), min(len(s), i + W + 1)):
            if i != j:
                C[idx[c], idx[s[j]]] += 1
print("vocab:", len(vocab), " nonzero cells:", int((C > 0).sum()))

## 2 · PPMI, then factorise

Raw counts are dominated by frequency. Positive pointwise mutual information normalises for
how common each concept is on its own, which is what makes the geometry meaningful.

In [ ]:
tot    = C.sum()
pr, pc = C.sum(1) / tot, C.sum(0) / tot
with np.errstate(divide='ignore', invalid='ignore'):
    PMI = np.log((C / tot) / np.outer(pr, pc))
PPMI = np.nan_to_num(np.maximum(PMI, 0), nan=0, posinf=0, neginf=0)

DIM = min(24, len(vocab) - 1)
E = pd.DataFrame(normalize(TruncatedSVD(DIM, random_state=0).fit_transform(PPMI)),
                 index=vocab)

def neighbours(tag, k=6):
    sim = E.values @ E.loc[tag].values
    return list(E.index[np.argsort(-sim)[1:k+1]])

for t in ['letrozole', 'metformin', 'chest_pain', 'clomiphene', 'ekg']:
    if t in E.index:
        print(f"{t:14s} -> {neighbours(t)}")

Look at `letrozole`'s neighbours. You will see **both** the fertility bundle (clomiphene,
gonadotropin, pelvic ultrasound) **and** the oncology bundle (anastrozole, mastectomy).

That is the punchline of the day. A static concept embedding has **one vector per concept**,
so it averages the two indications into a single position and cannot disambiguate them. It
is the bag-of-codes failure in a smarter coordinate system.

Fixing it requires a **contextual** representation — the same concept getting different
vectors depending on what surrounds it. That is precisely what a transformer over timelines
provides, and it is the real argument for CLMBR, stated far more precisely than "foundation
models are good."

## 3 · Patient vectors by pooling

In [ ]:
P_mean = normalize(np.vstack([E.loc[seqs[p]].values.mean(0) for p in pids]))

Pw = []
for p in pids:
    v = E.loc[seqs[p]].values
    w = np.exp(np.linspace(-2, 0, len(v)))          # recent events weigh more
    Pw.append((v * w[:, None]).sum(0) / w.sum())
P_recency = normalize(np.vstack(Pw))

X  = eo.bag_of_codes(events, min_count=20).reindex(pids).fillna(0)
Zb = TruncatedSVD(min(24, X.shape[1] - 1), random_state=0).fit_transform(
         normalize(X.values.astype(float)))

if y is not None:
    for name, M in [('bag-of-codes + SVD', Zb), ('embedding, mean pool', P_mean),
                    ('embedding, recency pool', P_recency)]:
        ari = adjusted_rand_score(y, KMeans(7, n_init=10, random_state=0).fit(M).labels_)
        print(f"{name:26s} ARI = {ari:.3f}")
else:
    from sklearn.metrics import silhouette_score
    for name, M in [('bag-of-codes + SVD', Zb), ('embedding, mean pool', P_mean),
                    ('embedding, recency pool', P_recency)]:
        l = KMeans(7, n_init=10, random_state=0).fit(M).labels_
        print(f"{name:26s} silhouette = {silhouette_score(M, l):.3f}")
    print("\nNo labels on real data -- compare via stability (Day 2) and downstream")
    print("outcome separation (Day 5), not via a single intrinsic score.")

Compare honestly. The embedding helps, but not dramatically — and the recency-weighted
version can do *worse*, because crude time weighting discards information rather than
modelling it.

**The lesson for the CLMBR conversation:** representation improvements are real but
incremental, and a poorly chosen time encoding actively hurts. "Add a transformer" is not a
plan; the time representation is the design decision that matters.

## 4 · Point-in-time truncation

Non-negotiable. Embed each patient **as of** an index date, using only prior events. Skip
this and every metric you report is leakage.

In [ ]:
index = eo.index_dates(events, 'letrozole')                  # first letrozole exposure
past  = eo.truncate_at(events, index)                        # events at or before it
seq_at = eo.sequences(past, col='tag')

lz_pids = [p for p in index.person_id if p in seq_at and seq_at[p]]
V = normalize(np.vstack([E.loc[[c for c in seq_at[p] if c in E.index]].values.mean(0)
                         for p in lz_pids]))
print("letrozole patients embedded at index:", V.shape)

weak = eo.weak_labels(events)
wl   = weak.set_index('person_id').weak_label
col  = [wl.get(p, 'unlabelled') for p in lz_pids]

from sklearn.manifold import TSNE
T = TSNE(2, init='pca', perplexity=min(12, max(5, len(V) // 6)),
         random_state=0).fit_transform(V)
plt.figure(figsize=(6.5, 5))
for name, g in pd.DataFrame({'x': T[:,0], 'y': T[:,1], 'c': col}).groupby('c'):
    plt.scatter(g.x, g.y, s=30, alpha=.8, label=name)
plt.legend(fontsize=8); plt.xticks([]); plt.yticks([])
plt.title('letrozole cohort embedded AT index date (no future leakage)')
plt.tight_layout(); plt.show()

Run it once **without** `truncate_at` and watch separation improve. That improvement is
entirely fake — after the index date, the oncology patients accumulate mastectomies and
anastrozole. Do it once deliberately so you recognise the signature of leakage in someone
else's results.

## 5 · From embeddings back to motifs — the bridge

Embeddings are not auditable. The fix is to cluster them and mine sequences *inside* each
cluster. The frequent sequences per cluster **are** the cluster's name.

This is the single most useful pattern in the whole week, because it produces both objects
at once: continuous vectors for the survival model, named motifs for the rule engine.

In [ ]:
import collections

lab = KMeans(7, n_init=10, random_state=0).fit(P_mean).labels_
db  = [seqs[p] for p in pids]

rows = []
for c in range(7):
    inn, out = collections.Counter(), collections.Counter()
    for i, s in enumerate(db):
        tgt = inn if lab[i] == c else out
        for u, v in zip(s, s[1:]):
            tgt[(u, v)] += 1
    ni, no = max(1, (lab == c).sum()), max(1, (lab != c).sum())
    scored = sorted(((inn[b] / ni + .01) / (out[b] / no + .01), b)
                    for b in inn if inn[b] >= 10)[::-1]
    rows.append({'cluster': c, 'n': int((lab == c).sum()),
                 'motif candidates': ' | '.join(f"{u}->{v}" for _, (u, v) in scored[:3])})
display(pd.DataFrame(rows))

## 6 · Where this hands off to CLMBR / MEDS

Your extract already writes standard-named symlinks so `meds_etl_omop` can find the files.
The MEDS conversion is therefore roughly:

```bash
pip install meds_etl
meds_etl_omop $OUT_DIR meds_out/     # $OUT_DIR = omop_extract/ or data_input/
```

Then `femr` tokenises `meds_out/` and CLMBR runs a frozen forward pass over it. Everything
you built today is that pipeline with the transformer swapped for an SVD — same inputs, same
output shape, different learner.

Two things to settle before that sprint, both raised in the review:

**The licence.** CLMBR-T-base weights are cc-by-nc-4.0, non-commercial only, gated behind a
credentialed health-data licence and a CITI certificate. Get a legal read before anyone
plans around them. Architecture and `femr` are usable regardless.

**Out-of-vocabulary rate.** The pretrained vocabulary is the top-k frequent codes from
hospital EHR data — labs, vitals, inpatient physiology. Your claims are HCPCS, CPT and NDC
heavy with none of that. Measure the OOV rate first; it decides whether transfer learning is
even available or whether pretraining from scratch is mandatory.

| Model | Objective | Why you would pick it |
|---|---|---|
| **CLMBR-T-base** | autoregressive next-code | OMOP-native, released weights, non-commercial |
| **MOTOR** | time-to-event pretraining | lines up with survival endpoints directly |
| **CEHR-BERT / Med-BERT** | masked code prediction | CEHR-BERT has explicit time tokens |

## Exercises

1. Vary `W` from 1 to 10. How do `letrozole`'s neighbours change?
2. Split `letrozole` into two pseudo-tags by whether a cancer diagnosis precedes it, then
   re-embed. Do the neighbour lists separate? This hand-builds contextualisation — and is a
   cheap proxy for what a transformer would do.
3. Replace mean pooling with max pooling and re-score.
4. Compute the OOV rate directly: what fraction of your `token` values are LOINC or HCPCS
   codes that an EHR-pretrained vocabulary is unlikely to contain?

## Carry forward

Two representations, a clustering over each, and a motif catalog.